# 2018–2022韩国海关前20日：官方页面及原附件原字节抓取

冻结60个页面ID并取得其所有可识别HWP/HWPX/PDF附件；按年份归档NAS。仅来源，没有半导体因子与SOXX结果。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex,time,urllib.request,urllib.parse,urllib.error,datetime,re
import pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2018_2022_original_capture_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
index=root/src['index_card'];old=index/'qlib_recorder'/src['index_experiment_id']/src['index_recorder_id']/'artifacts/analysis/repaired_index';assert sha(index/'summary.json')==src['index_summary_sha256'] and sha(old)==src['index_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.uri_context(str(index/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['index_experiment_id'],recorder_id=src['index_recorder_id']);assert rec.status=='FINISHED';monthly=rec.load_object('analysis/repaired_index')
assert len(monthly)==60 and monthly.newsId.notna().all() and monthly.month.tolist()==[f'{y}-{m:02d}' for y in range(2018,2023) for m in range(1,13)]
snapshot=root/'data/snapshots/kcs_early20_2018_2022_original_capture_20261023';snapshot.mkdir(parents=True,exist_ok=True);transport=plan['transport'];state={'status':'source_capture_started','plan_sha256':sha(card/'plan.json'),'page_attempts':0,'attachment_attempts':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
def fetch(url,maxbytes):
 started=datetime.datetime.now(datetime.timezone.utc).isoformat();headers={};body=b'';status=None;error=None;final=url
 try:
  response=urllib.request.urlopen(urllib.request.Request(url,headers={'User-Agent':transport['user_agent']}),timeout=transport['timeout_seconds_each']);status=response.status;final=response.url;headers={k:v for k,v in response.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=response.read(maxbytes+1);response.close()
 except urllib.error.HTTPError as exc:
  status=exc.code;final=exc.url;headers={k:v for k,v in exc.headers.items() if k.lower() not in ('set-cookie','set-cookie2','authorization','proxy-authenticate')};body=exc.read(maxbytes+1);error='HTTPError'
 except Exception as exc:error=type(exc).__name__+': '+str(exc)[:180]
 return {'url':url,'final_url':final,'started_utc':started,'ended_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'http_status':status,'content_type':headers.get('Content-Type',headers.get('content-type','')),'body_sha256':hashlib.sha256(body).hexdigest(),'body_bytes':len(body),'error':error,'headers':headers},body
def save(year,key,kind,receipt,body):
 folder=snapshot/str(year);folder.mkdir(parents=True,exist_ok=True);path=folder/(key+'.'+kind+'.response');path.write_bytes(body);head=folder/(key+'.'+kind+'.headers.json');head.write_text(json.dumps(receipt.pop('headers'),ensure_ascii=False,indent=2)+'\n');receipt['local_path']=str(path.relative_to(root));receipt['headers_path']=str(head.relative_to(root));receipt['headers_sha256']=sha(head);return receipt
receipts=[];rows=[]
for row in monthly.itertuples():
 key=row.month;year=int(key[:4]);newsid=str(row.newsId);page_url=transport['page_url_template'].replace('{newsId}',newsid);state['page_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');page,body=fetch(page_url,transport['max_page_bytes']);page=save(year,key,'page',page,body);page['newsId']=newsid;page['month']=key;page_status='unknown';attachments=[];links=[]
 if page['http_status']==200 and page['body_bytes']<=transport['max_page_bytes'] and 'html' in page['content_type'].lower():
  soup=BeautifulSoup(body,'html.parser');titles=[h.get_text(' ',strip=True) for h in soup.find_all('h1')];compact=[re.sub(r'\s+','',t) for t in titles];page_status='identified' if any(('수출입' in t and '20일' in t) or (f'{int(key[-2:])}.1' in t and '20' in t and '수출입' in t) for t in compact) else 'title_unknown';page['heading_candidates']=titles[:5]
  if page_status=='identified':
   seen=set()
   for a in soup.find_all('a'):
    href=a.get('href') or '';name=a.get_text(' ',strip=True);m=re.search(r'\.(hwp|hwpx|pdf)\b',name,re.I);fileid=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('fileId') or [None])[0]
    if 'download.do' not in href or m is None or fileid is None or fileid in seen:continue
    seen.add(fileid);links.append({'fileId':fileid,'filename':name,'ext':m.group(1).lower(),'url':urllib.parse.urljoin(page_url,href)})
 if page_status=='identified' and len(links)<=transport['max_attachments_per_page']:
  for link in links:
   state['attachment_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');a,data=fetch(link['url'],transport['max_attachment_bytes']);a=save(year,key+'_'+link['fileId'],'attachment',a,data);a.update(link);expected={'hwp':bytes.fromhex('d0cf11e0a1b11ae1'),'hwpx':b'PK\x03\x04','pdf':b'%PDF'}[link['ext']];a['signature_ok']=bool(a['http_status']==200 and a['body_bytes']<=transport['max_attachment_bytes'] and data.startswith(expected));attachments.append(a);time.sleep(transport['rate_pause_seconds'])
 else:
  if len(links)>transport['max_attachments_per_page']:page_status='attachment_overflow_unknown'
 page['page_status']=page_status;page['attachment_links']=links;page['attachments']=attachments;receipts.append(page);rows.append({'month':key,'newsId':newsid,'page_status':page_status,'attachment_links':len(links),'attachment_attempts':len(attachments),'signature_qualified_attachments':sum(x['signature_ok'] for x in attachments),'all_linked_signatures_qualified':bool(links and len(links)==len(attachments) and all(x['signature_ok'] for x in attachments))});print('captured',key,page_status,'links',len(links),'qualified',rows[-1]['signature_qualified_attachments'],flush=True);time.sleep(transport['rate_pause_seconds'])
assert state['page_attempts']==60 and state['attachment_attempts']<=plan['budget']['attachment_GETs_max'];state['status']='source_GETs_finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
settings=_settings(root);archives=[]
for year in range(2018,2023):
 folder=snapshot/str(year);year_rows=[x for x in receipts if x['month'].startswith(str(year))];assert len(year_rows)==12;year_manifest={'plan_sha256':sha(card/'plan.json'),'year':year,'months':year_rows};(folder/'year_manifest.json').write_text(json.dumps(year_manifest,ensure_ascii=False,indent=2)+'\n');files=sorted(p for p in folder.iterdir() if p.is_file())
 with tempfile.TemporaryDirectory(prefix='kcs-original-'+str(year)+'-') as temp:
  archive=pathlib.Path(temp)/'source.tar.gz'
  with tarfile.open(archive,'w:gz') as t:
   for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
  digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
  remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest;archives.append({**receipt,'year':year,'nas_path':remote,'files':len(files),'year_manifest_sha256':sha(folder/'year_manifest.json')})
  with tempfile.TemporaryDirectory(prefix='kcs-original-cold-'+str(year)+'-') as tmp:
   copy=pathlib.Path(tmp)/'source.tar.gz'
   with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,check=True)
   assert copy.stat().st_size==receipt['size_bytes'] and sha(copy)==digest
   with tarfile.open(copy,'r:gz') as t:
    names={m.name for m in t if m.isfile()};assert names=={str(p.relative_to(root)) for p in files}
    for p in files:assert t.extractfile(str(p.relative_to(root))).read()==p.read_bytes()
 print('NAS year cold restored',year,len(files),flush=True)
table=pd.DataFrame(rows);counts=table.page_status.value_counts().to_dict();complete=int(table.all_linked_signatures_qualified.sum());result={'status':'official_2018_2022_early20_original_page_attachment_capture_source_only','plan_sha256':sha(card/'plan.json'),'months':60,'page_GET_attempts':state['page_attempts'],'attachment_GET_attempts':state['attachment_attempts'],'page_status_counts':counts,'months_all_linked_signatures_qualified':complete,'months_without_qualified_attachment':table.loc[~table.all_linked_signatures_qualified,'month'].tolist(),'year_archives':archives,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
restore={'status':'NAS_five_year_original_groups_all_members_cold_restored_exact','years':[{'year':x['year'],'sha256':x['sha256'],'files':x['files']} for x in archives]};(card/'source_restore.json').write_text(json.dumps(restore,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,index_identity={'summary_sha256':src['index_summary_sha256'],'artifact_sha256':src['index_artifact_sha256']},year_archives=archives);rec.save_objects(artifact_path='analysis',monthly_capture=table,source_receipts=receipts,result=result);rec.log_metrics(months=60,page_GET_attempts=state['page_attempts'],attachment_GET_attempts=state['attachment_attempts'],months_all_linked_signatures_qualified=complete,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');rec_archive=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(rec_archive['nas_path'])).split()[0]==rec_archive['sha256']
summary={'experiment':plan['experiment'],'decision':'KCS_2018_2022_original_attachment_source_capture_numeric_PIT_unqualified','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','year_archives':archives,'source_restore':'source_restore.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':rec_archive,'NAS_readback':{'sha256':rec_archive['sha256'],'verified':True},'page_GET_attempts':state['page_attempts'],'attachment_GET_attempts':state['attachment_attempts'],'months_all_linked_signatures_qualified':complete,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_capture','analysis/source_receipts','analysis/result']},'limitations':['Current attachments and portal dates do not prove exact historical first-public versions.','No semiconductor amount/YoY was parsed, aligned into Qlib, fit or evaluated against SOXX.','Any transport/signature gap stays unknown and needs a separately frozen repair.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');print(json.dumps({'decision':summary['decision'],'page_status_counts':counts,'qualified_months':complete,'unknown_months':result['months_without_qualified_attachment'],'attachment_GETs':state['attachment_attempts'],'year_archives':[x['sha256'] for x in archives],'recorder_id':rid,'Recorder_NAS_sha256':rec_archive['sha256']},ensure_ascii=False,indent=2))


## 中文结论

按修复后的固定60行官方页面索引逐月抓取，每页一次、不重试。**60/60 页面可辨认，页面共列出71份附件：60份HWP、11份PDF**；所有71份均由该页的唯一 `fileId` 下载且响应签名符合文件类型，11个月同时列出HWP/PDF，两版均保留，不预先选定数值版本。页面和附件响应、头、UTC收取时间、URL、SHA及失败状态都保存为原字节；本轮网页GET 60次、附件GET 71次。

五个年度源压缩包已写入 Yanxuan NAS 并**逐文件冷恢复**，源哈希与逐年成员数见 [summary.json](summary.json)；原生Qlib Recorder `a02d48c6705e444bb35d1266f79e96ce` 保存完整60行状态和所有来源收据，NAS SHA-256 `bda4cc80a07b08dbec1ba3a5f69071bff24fb217ce7b5e71b8ffc0f97a261675` 的三件索引/回执/结果工件也限定冷读一致。2021年10月HWP/PDF与2022年10月HWP再次抓到的字节，逐份与此前各自归档的原件哈希完全相同；这只排除这三份**当前版本**发生变动，不能倒推历史首发字节。

决定 `KCS_2018_2022_original_attachment_source_capture_numeric_PIT_unqualified`。**原件传输覆盖不等于可交易信息面**：尚未逐月核表格里的半导体金额、同比、工作日差异与HWP/PDF版本一致性；官方页面的发布日期和今天可下载的字节也不足以认证当时首次在线版本或美股决策前可用性。必须先做固定数值/版本/时钟审查，再映射到 Qlib 与 Stage1 原39项匹配接收器。此卡0 SOXX收益读取、0拟合、0账户和0合格策略前向日；SOXX稳定超额仍未完成。
